# Regional VMD EOG removal — vmd-diagnosis

Inspect solver reconstruction, projection closure, mode gates and calibration baselines.

## Scientific question, inputs and contracts

Required parents: completed corpus and VMD search. Reuse immutable selected-mode vectors; do not refit or overwrite the search. Compare exact mode-plus-residual reconstruction and linear projection closure against matched direct regression. Partition recovery MSE into squared mean bias and centered variance; centered-only SNR is never an acceptance metric. Test an explicit unscored EOG calibration baseline as an experimental alternative to zero-mean scoring projection. Clean targets and true artifact are diagnostic-only; no target enters the correction. This Fz-focused mechanism study cannot authorize a final teacher or model.

## Fresh source and environment
Resolve latest published code once and pin it. New temporary clones avoid stale Python imports; original inputs remain immutable.

In [ ]:
import os, sys, json, tempfile, subprocess
from pathlib import Path
IN_KAGGLE = Path('/kaggle/input').is_dir()
if not IN_KAGGLE:
    raise RuntimeError('This numerical campaign runs on Kaggle only')
TEMP_ROOT = Path(tempfile.mkdtemp(prefix='vmd-eog-'))
CHECKOUT = TEMP_ROOT / 'repository'
REPOSITORY_URL = 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git'
BRANCH = 'research/vmd-eog-removal-notebooks'
RUN_SPEC = {'stage': 'vmd-diagnosis', 'run_id': 'vmd-diagnosis-manual', 'git_sha': '', 'campaign_id': 'vmd-eog-20261010', 'profile': 'pilot'}
# Manual launches resolve latest once; CLI launches already have its exact SHA.
if not RUN_SPEC['git_sha']:
    line = subprocess.check_output(['git','ls-remote',REPOSITORY_URL,'refs/heads/'+BRANCH], text=True).strip()
    RUN_SPEC['git_sha'] = line.split()[0]
subprocess.check_call(['git','clone','--quiet','--branch',BRANCH,'--single-branch',REPOSITORY_URL,str(CHECKOUT)])
subprocess.check_call(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']])
ACTUAL_SHA = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if ACTUAL_SHA != RUN_SPEC['git_sha']:
    raise RuntimeError('Source SHA mismatch')
ENV_ROOT = TEMP_ROOT / 'environment'
subprocess.check_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV_ROOT)])
ENV_PY = ENV_ROOT / 'bin/python'
subprocess.check_call([sys.executable,'-m','pip','--python',str(ENV_PY),'install','--quiet','--progress-bar','off','-r',str(CHECKOUT/'requirements.txt')])
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONPATH'] = str(CHECKOUT/'src')
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
PROCESS_ENV['OMP_NUM_THREADS'] = '2'
OUT = Path('/kaggle/working/results') / RUN_SPEC['campaign_id'] / RUN_SPEC['run_id']
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
print('Exact Git SHA:', ACTUAL_SHA, 'Stage:', RUN_SPEC['stage'], 'Output:',OUT,flush=True)


## Inspect the pinned algorithm contract
The source modules remain inspectable in this checkout. Mode vectors, references and paired targets have distinct roles; the accompanying explanation defines their shapes and inference boundaries.

In [ ]:
CONFIGURATION = json.loads((CHECKOUT/'configs/campaign.json').read_text())
print('Declared signal contract:', {key: CONFIGURATION[key] for key in ('fs','window','hop','seed')})
print('Preservation limits:', CONFIGURATION['preservation'])
print('Algorithm explanation: docs/SCIENTIFIC_EXPLANATION.md in the pinned checkout')
print('Modules: signal.py -> reference.py/frontal.py/posterior.py -> experiments.py -> review evidence')
print('Git source:', ACTUAL_SHA)


## Execute reusable modules
HEOG/VEOG are teacher/evaluation information. Clean targets cannot enter inference. Failed numerical or scientific gates stop dependent work.

In [ ]:
command = [str(ENV_PY),'-m','vmd_eog.runner','--stage',RUN_SPEC['stage'],
           '--input','/kaggle/input','--output',str(OUT),'--profile',RUN_SPEC['profile']]
process = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
if process.returncode:
    raise RuntimeError('Stage failed; inspect saved execution_state.json and kernel logs')
print((OUT/'execution_state.json').read_text())


## Saved configurations and result tables
These are saved outputs from this run, not variables inherited from another notebook. Search tables describe development selection; the classical gate decides whether model work is authorized.

In [ ]:
import csv
from itertools import islice
for name in ('vmd_search.csv','posterior_search.csv','review_methods.csv','condition_source_uncertainty.csv','paper_metric_fixture.csv','paper_paired_channels.csv','paper_native_channels.csv','paper_rest_bandpower_channels.csv','paper_paired_source_means.csv','paper_native_source_means.csv','paper_paired_permutation_tests.csv','paper_native_permutation_tests.csv','native_condition_channels.csv','native_condition_participants.csv','native_condition_permutations.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 8 saved rows; complete table remains in outputs)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),8)))
for name in ('selected_frontal.json','selected_posterior.json','classical_gate.json','paper_evaluation_protocol.json','paper_metric_fixture_summary.json'):
    if (OUT/name).exists():
        print(name, (OUT/name).read_text()[:8000])
review = OUT/'APPROACH_REVIEW.md'
if review.exists():
    from IPython.display import Markdown
    display(Markdown(review.read_text()))


## Inspect evidence and persist outputs
Native real EEG has no clean-reference SNR. Pass-through failures remain counted. Parameters and physiological frequency bands are distinct.

In [ ]:
from IPython.display import display, Image
for figure in sorted(OUT.glob('*.png')):
    display(Image(filename=str(figure)))
for report in sorted(OUT.glob('*summary.json')):
    print(report.name, report.read_text()[:16000])
print('Persist this completed kernel version and retrieve outputs before launching a dependent stage.')
